In [41]:
# 4(a): Pull in Boston housing data

import pandas as pd
from keras.datasets import boston_housing

(train_data, train_targets), (test_data, test_targets) = boston_housing.load_data()
print(boston_housing.load_data.__doc__)

Loads the Boston Housing dataset.

    This is a dataset taken from the StatLib library which is maintained at
    Carnegie Mellon University.

    **WARNING:** This dataset has an ethical problem: the authors of this
    dataset included a variable, "B", that may appear to assume that racial
    self-segregation influences house prices. As such, we strongly discourage
    the use of this dataset, unless in the context of illustrating ethical
    issues in data science and machine learning.

    Samples contain 13 attributes of houses at different locations around the
    Boston suburbs in the late 1970s. Targets are the median values of
    the houses at a location (in k$).

    The attributes themselves are defined in the
    [StatLib website](http://lib.stat.cmu.edu/datasets/boston).

    Args:
        path: path where to cache the dataset locally
            (relative to `~/.keras/datasets`).
        test_split: fraction of the data to reserve as test set.
        seed: Random seed

In [42]:
# 4(b) Convert into Dataframes
# Data was already split into training and test samples

def data_to_dataframe(data, targets):
  df = pd.DataFrame(
      data,
      columns = [f"Feature{n}" for n in range(1,14)]
  )
  df.loc[:,"Label"] = targets
  return df

train_df = data_to_dataframe(train_data, train_targets)
test_df = data_to_dataframe(test_data, test_targets)

train_df.describe()

,Feature1,Feature2,Feature3,Feature4,Feature5,Feature6,Feature7,Feature8,Feature9,Feature10,Feature11,Feature12,Feature13,Label
count,404.000000,404.000000,404.000000,404.000000,404.000000,404.000000,404.000000,404.000000,404.000000,404.000000,404.000000,404.000000,404.000000,404.000000
mean,3.745111,11.480198,11.104431,0.061881,0.557356,6.267082,69.010644,3.740271,9.440594,405.898515,18.475990,354.783168,12.740817,22.395050
std,9.240734,23.767711,6.811308,0.241238,0.117293,0.709788,27.940665,2.030215,8.698360,166.374543,2.200382,94.111148,7.254545,9.210442
min,0.006320,0.000000,0.460000,0.000000,0.385000,3.561000,2.900000,1.129600,1.000000,188.000000,12.600000,0.320000,1.730000,5.000000
25%,0.081437,0.000000,5.130000,0.000000,0.453000,5.874750,45.475000,2.077100,4.000000,279.000000,17.225000,374.672500,6.890000,16.675000
50%,0.268880,0.000000,9.690000,0.000000,0.538000,6.198500,78.500000,3.142300,5.000000,330.000000,19.100000,391.250000,11.395000,20.750000
75%,3.674808,12.500000,18.100000,0.000000,0.631000,6.609000,94.100000,5.118000,24.000000,666.000000,20.200000,396.157500,17.092500,24.800000
max,88.976200,100.000000,27.740000,1.000000,0.871000,8.725000,100.000000,10.710300,24.000000,711.000000,22.000000,396.900000,37.970000,50.000000


In [56]:
#Scaling is only strictly necessary for Gradient Descent,
#but I'm scaling the data in advance for consistency, and
#because the instructions said to use scaled data for OLS too

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler().fit(train_df)
scaled_train_df = pd.DataFrame(
    scaler.transform(train_df),
    columns= train_df.columns
)
scaled_training_features = scaled_train_df.drop(columns=["Label"])
scaled_training_labels = scaled_train_df.loc[:, "Label"]

scaled_test_df = pd.DataFrame(
    scaler.transform(test_df),
    columns= test_df.columns
)
scaled_test_features = scaled_test_df.drop(columns=["Label"])
scaled_test_labels = scaled_test_df.loc[:, "Label"]

In [57]:
# 4(c) Train Ordinary Least Squares model and show results

from sklearn.linear_model import LinearRegression

ols_model = LinearRegression().fit(scaled_training_features, scaled_training_labels)
print(f"Weights: {ols_model.coef_}")
print(f"Bias: {ols_model.intercept_}")

ols_training_r2 = ols_model.score(scaled_training_features, scaled_training_labels)
print(f"R^2 on training data: {ols_training_r2}")

ols_test_r2 = ols_model.score(scaled_test_features, scaled_test_labels)
print(f"R^2 on test data: {ols_test_r2}")

Weights: [-0.12039218  0.14709038  0.0029461   0.10809323 -0.26106711  0.26049337
  0.02295841 -0.37734568  0.31613628 -0.21278523 -0.2155645   0.08909096
 -0.43780576]
Bias: 4.551709260629251e-16
R^2 on training data: 0.7399643695249463
R^2 on test data: 0.7213535934621551


In [58]:
# 4(d) Train Stochastic Gradient Descent model and show results

from sklearn.linear_model import SGDRegressor
from sklearn.pipeline import make_pipeline

sgd_model = make_pipeline(
    # Data has already been scaled; not necessary to scale again here
    SGDRegressor(max_iter= 10000, tol=1e-3, learning_rate='adaptive')
).fit(scaled_training_features, scaled_training_labels)
sgd_step = sgd_model.named_steps['sgdregressor']
print(f"Weights: {sgd_step.coef_}")
print(f"Bias: {sgd_step.intercept_}")

sgd_training_r2 = sgd_model.score(scaled_training_features, scaled_training_labels)
print(f"R^2 on training data: {sgd_training_r2}")

sgd_test_r2 = sgd_model.score(scaled_test_features, scaled_test_labels)
print(f"R^2 on test data: {sgd_test_r2}")

Weights: [-0.11951463  0.14318622 -0.00419003  0.10673701 -0.25801871  0.26542542
  0.01775432 -0.38160349  0.29240517 -0.18924607 -0.21430698  0.09033675
 -0.43407942]
Bias: [-0.00112061]
R^2 on training data: 0.7398696375095899
R^2 on test data: 0.7227436961030143


In [60]:
# 4(e) Present results as a 2x2 dataframe

def adjusted_r2(r2, data_df):
  num_samples = data_df.shape[0]
  num_features = data_df.shape[1]
  return 1 - (1-r2) * (num_samples - 1) / (num_samples - num_features - 1)

df = pd.DataFrame(
    [
        [ols_test_r2, adjusted_r2(ols_test_r2, train_data)],
        [sgd_test_r2, adjusted_r2(sgd_test_r2, train_data)]
    ],
    index= ["Ordinary Least Squares", "Stochastic Gradient Descent"],
    columns= ["Test r^2", "Adjusted r^2"]
)
print(df)

                             Test r^2  Adjusted r^2
Ordinary Least Squares       0.721354      0.712065
Stochastic Gradient Descent  0.722744      0.713502
